# Spark Architecture and Data Processing Assignment


**Objective:** Learn Spark architecture and do basic data processing on a sales dataset.

## 1. Introduction to Apache Spark

Apache Spark is a tool for processing large data on many machines. It can do fast analytics and data transformations.

Spark is usually faster than Hadoop MapReduce because it keeps intermediate data in memory instead of writing every step to disk.

### Spark Core
Spark Core is the engine that runs tasks and manages memory.

### Spark SQL
Spark SQL is used for working with structured data and tables.

### DataFrame API
DataFrames are like tables and let you select, filter, and group data.

### RDD
RDD is the older Spark data structure. DataFrames are easier for most jobs.

### SparkSession
SparkSession is the starting point for Spark in Python.

## 2. Spark Architecture

Spark has a Driver, a Cluster Manager, and Executors. The Driver coordinates the job, the Cluster Manager gives resources, and Executors do the work.

### Driver
The Driver controls the application and creates SparkSession.

### Cluster Manager
The Cluster Manager gives CPUs and memory to Spark.

### Executors
Executors run tasks and process data.

### Execution Modes
- Local Mode: run on one machine.
- Standalone: Spark runs on its own cluster.
- YARN: Spark runs on Hadoop YARN.
- Kubernetes: Spark runs on Kubernetes.

### Architecture Diagram
```
User
   |
Spark Driver
   |
Cluster Manager
   |
-----------------------
|          |          |
Executor Executor Executor
```

## 3. Lazy Evaluation

Spark does not run transformations right away. It waits for actions like `show()` or `count()`. This helps Spark optimize work before execution.

### Transformations
- `select()`
- `filter()`
- `withColumn()`

### Actions
- `show()`
- `count()`
- `collect()`


## 5. Create Spark Session

Start Spark in local mode using SparkSession.

In [94]:
import os
import urllib.request
from pyspark.sql import SparkSession

# Fix for Windows - download Hadoop utilities needed for Parquet
hadoop_home = 'C:\\hadoop'
hadoop_bin = os.path.join(hadoop_home, 'bin')
os.makedirs(hadoop_bin, exist_ok=True)

winutils_file = os.path.join(hadoop_bin, 'winutils.exe')

# Download if not exists
if not os.path.exists(winutils_file):
    print("Downloading Hadoop utilities...")
    url = "https://github.com/kontext-tech/winutils/raw/master/hadoop-3.3.1/bin/winutils.exe"
    urllib.request.urlretrieve(url, winutils_file)
    print("Downloaded successfully!")

# Set up environment
os.environ['HADOOP_HOME'] = hadoop_home
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# Create Spark session
spark = SparkSession.builder \
    .appName('SparkArchitectureIntern') \
    .master('local[*]') \
    .getOrCreate()

spark

## 6. Demonstrate Lazy Evaluation

We define transformations first and run them later with actions.

In [95]:
# Step 1: Read the CSV file
csv_file = 'd:/celebal technologies/week6/Assignment/dataset/sales_data.csv'
sales_df = spark.read.csv(csv_file, header=True, inferSchema=True)

# Step 2: Pick only the columns we need
sales_df = sales_df.select('Order_ID', 'Order_Date', 'Sales', 'Discount')

# Step 3: Keep only rows where Sales > 1000 AND Discount > 0
sales_df = sales_df.filter(sales_df['Sales'] > 1000)
sales_df = sales_df.filter(sales_df['Discount'] > 0.0)

# Step 4: Create a new column called 'Net_Sales' (Price after discount)
sales_df = sales_df.withColumn('Net_Sales', sales_df['Sales'] * (1 - sales_df['Discount']))

# Step 5: Show first 5 rows and count total rows
sales_df.show(5, truncate=False)
print('Total rows:', sales_df.count())

+---------+----------+-------+--------+------------------+
|Order_ID |Order_Date|Sales  |Discount|Net_Sales         |
+---------+----------+-------+--------+------------------+
|ORD000002|2023-10-21|1845.72|0.1     |1661.1480000000001|
|ORD000003|2023-03-31|2465.02|0.1     |2218.518          |
|ORD000005|2023-03-26|2980.62|0.1     |2682.558          |
|ORD000009|2023-05-06|1499.2 |0.2     |1199.3600000000001|
|ORD000010|2023-10-06|2328.49|0.1     |2095.641          |
+---------+----------+-------+--------+------------------+
only showing top 5 rows

Total rows: 6482


## 7. Read CSV File

Read the generated CSV file with inferred schema.

In [96]:
sales_df = spark.read.csv(csv_file, header=True, inferSchema=True)
sales_df.printSchema()
sales_df.show(5, truncate=False)


root
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)

+---------+----------+-----------+-------------+------+----------+-------------+---------------+------------+------------+--------+-------+--------+-------+
|Order_ID |Order_Date|Customer_ID|Customer_Name|Region|State     |City         |Category       |Sub_Category|Product_Name|Quantity|Sales  |Discount|Profit |
+---------+----------+-----------+-------------+------+----------+-------------+---------------+------------+--------

## 8. Read CSV using Manual Schema

Use a manual schema so Spark does not need to guess data types.

In [97]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

manual_schema = StructType([
    StructField('Order_ID', StringType(), True),
    StructField('Order_Date', DateType(), True),
    StructField('Customer_ID', StringType(), True),
    StructField('Customer_Name', StringType(), True),
    StructField('Region', StringType(), True),
    StructField('State', StringType(), True),
    StructField('City', StringType(), True),
    StructField('Category', StringType(), True),
    StructField('Sub_Category', StringType(), True),
    StructField('Product_Name', StringType(), True),
    StructField('Quantity', IntegerType(), True),
    StructField('Sales', DoubleType(), True),
    StructField('Discount', DoubleType(), True),
    StructField('Profit', DoubleType(), True),
])

sales_manual = spark.read.csv(csv_file, header=True, schema=manual_schema, dateFormat='yyyy-MM-dd')
sales_manual.printSchema()
sales_manual.show(5, truncate=False)


root
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)

+---------+----------+-----------+-------------+------+----------+-------------+---------------+------------+------------+--------+-------+--------+-------+
|Order_ID |Order_Date|Customer_ID|Customer_Name|Region|State     |City         |Category       |Sub_Category|Product_Name|Quantity|Sales  |Discount|Profit |
+---------+----------+-----------+-------------+------+----------+-------------+---------------+------------+--------

## 9. Convert CSV to Parquet

Write the dataset to Parquet and read it back.

In [98]:
# Display the processed DataFrame
print("Pipeline Result")

result_df.printSchema()
result_df.show(10, truncate=False)

Pipeline Result
root
 |-- Region: string (nullable = true)
 |-- Total_Sales: double (nullable = true)
 |-- Average_Sales: double (nullable = true)
 |-- Total_Profit: double (nullable = true)

+------+-----------------+------------------+------------------+
|Region|Total_Sales      |Average_Sales     |Total_Profit      |
+------+-----------------+------------------+------------------+
|West  |6191207.510000003|3008.3612779397486|277982.3599999994 |
|North |6282707.84       |3033.6590246257847|296948.22000000015|
|East  |5892451.419999997|2974.4833013629463|287560.22000000003|
|South |5999833.690000001|3027.161296670031 |281903.16999999963|
+------+-----------------+------------------+------------------+



## 10. DataFrame Operations

Use select, filter, rename, cast, and add columns.

In [99]:
# Select specific columns
result1 = parquet_df.select('Customer_Name', 'Sales', 'Profit')
print("Select columns:")
result1.show(5, truncate=False)

# Filter: Keep only West region with Sales > 1000
result2 = parquet_df.filter(parquet_df['Region'] == 'West')
result2 = result2.filter(parquet_df['Sales'] > 1000)
print("\nFilter - West region with Sales > 1000:")
result2.show(5, truncate=False)

# Rename: Change 'Sales' to 'Total_Sales'
result3 = parquet_df.withColumnRenamed('Sales', 'Total_Sales')
print("\nRenamed column:")
result3.select('Customer_Name', 'Total_Sales', 'Profit').show(5, truncate=False)

# Add new column: Calculate Net_Sales
result4 = parquet_df.withColumn('Net_Sales', parquet_df['Sales'] * (1 - parquet_df['Discount']))
print("\nNew column - Net_Sales:")
result4.select('Order_ID', 'Sales', 'Discount', 'Net_Sales').show(5, truncate=False)

Select columns:
+-------------+-------+-------+
|Customer_Name|Sales  |Profit |
+-------------+-------+-------+
|Customer 58  |834.15 |42.58  |
|Customer 236 |1845.72|18.88  |
|Customer 12  |2465.02|-39.79 |
|Customer 453 |3902.09|805.28 |
|Customer 68  |2980.62|-101.89|
+-------------+-------+-------+
only showing top 5 rows


Filter - West region with Sales > 1000:
+---------+----------+-----------+-------------+------+----------+-------------+---------------+------------+------------+--------+-------+--------+-------+
|Order_ID |Order_Date|Customer_ID|Customer_Name|Region|State     |City         |Category       |Sub_Category|Product_Name|Quantity|Sales  |Discount|Profit |
+---------+----------+-----------+-------------+------+----------+-------------+---------------+------------+------------+--------+-------+--------+-------+
|ORD000003|2023-03-31|CUST9415   |Customer 12  |West  |Georgia   |Miami        |Technology     |Phones      |Printer     |15      |2465.02|0.1     |-39.79 |
|O

## 11. Handle Null Values

Show how to drop nulls, fill nulls, and replace values.

In [100]:
# Drop rows with NULL values
print("Original rows:", parquet_df.count())
clean_df = parquet_df.dropna()
print("Rows after removing NULL values:", clean_df.count())

# Fill NULL values with default values
filled_df = parquet_df.fillna({'Discount': 0.0, 'Profit': 0.0})
print("\nAfter filling NULL values:")
filled_df.select('Customer_Name', 'Discount', 'Profit').show(5, truncate=False)

# Show summary statistics
print("\nSummary statistics:")
parquet_df.describe('Sales', 'Profit', 'Discount').show()

Original rows: 10000
Rows after removing NULL values: 10000

After filling NULL values:
+-------------+--------+-------+
|Customer_Name|Discount|Profit |
+-------------+--------+-------+
|Customer 58  |0.15    |42.58  |
|Customer 236 |0.1     |18.88  |
|Customer 12  |0.1     |-39.79 |
|Customer 453 |0.0     |805.28 |
|Customer 68  |0.1     |-101.89|
+-------------+--------+-------+
only showing top 5 rows


Summary statistics:
+-------+------------------+------------------+-------------------+
|summary|             Sales|            Profit|           Discount|
+-------+------------------+------------------+-------------------+
|  count|             10000|             10000|              10000|
|   mean| 2535.654236000003|119.85382999999993|0.09993499999999987|
| stddev|1426.0754798541327|430.12808385203437|0.07081488014340617|
|    min|             50.05|           -970.29|                0.0|
|    max|           4999.82|           1477.64|                0.2|
+-------+----------------

## 12. Transformations

Examples of common DataFrame transformations.

In [101]:
# Select specific columns
print("SELECT columns (Order_ID, Region, Sales):")
parquet_df.select('Order_ID', 'Region', 'Sales').show(3, truncate=False)

# Filter rows where Sales > 1000
print("\nFILTER rows (Sales > 1000):")
parquet_df.filter(parquet_df['Sales'] > 1000).show(3, truncate=False)

# Add new column
print("\nADD column (Net_Sales):")
parquet_df.withColumn('Net_Sales', parquet_df['Sales'] * (1 - parquet_df['Discount'])).select('Order_ID', 'Net_Sales').show(3, truncate=False)

# Remove column
print("\nDROP column (City):")
parquet_df.drop('City').show(3, truncate=False)

# Remove duplicate rows
print("\nREMOVE duplicates (Order_ID):")
parquet_df.dropDuplicates(['Order_ID']).show(3, truncate=False)

# Show distinct regions
print("\nDISTINCT Regions:")
parquet_df.select('Region').distinct().show(5, truncate=False)

# Sort by Sales (descending)
print("\nSORT by Sales (highest first):")
parquet_df.orderBy(parquet_df['Sales'].desc()).show(3, truncate=False)

SELECT columns (Order_ID, Region, Sales):
+---------+------+-------+
|Order_ID |Region|Sales  |
+---------+------+-------+
|ORD000001|South |834.15 |
|ORD000002|East  |1845.72|
|ORD000003|West  |2465.02|
+---------+------+-------+
only showing top 3 rows


FILTER rows (Sales > 1000):
+---------+----------+-----------+-------------+------+----------+-------------+----------+------------+------------+--------+-------+--------+------+
|Order_ID |Order_Date|Customer_ID|Customer_Name|Region|State     |City         |Category  |Sub_Category|Product_Name|Quantity|Sales  |Discount|Profit|
+---------+----------+-----------+-------------+------+----------+-------------+----------+------------+------------+--------+-------+--------+------+
|ORD000002|2023-10-21|CUST8456   |Customer 236 |East  |California|San Francisco|Technology|Phones      |Smartphone  |14      |1845.72|0.1     |18.88 |
|ORD000003|2023-03-31|CUST9415   |Customer 12  |West  |Georgia   |Miami        |Technology|Phones      |Printer

## 13. Actions

Actions run the Spark job and return results. Use them carefully.

In [102]:
# Show first 5 rows
print("Show first 5 rows:")
parquet_df.show(5, truncate=False)

# Count total rows
print("Total rows:", parquet_df.count())

# Get first 5 rows as list
print("\nGet first 5 rows as list:")
sample = parquet_df.limit(5).collect()
print("Sample size:", len(sample))

# Get first row
print("\nFirst row:")
print(parquet_df.first())

# Get statistics
print("\nStatistics for Sales and Profit:")
parquet_df.describe('Sales', 'Profit').show()

Show first 5 rows:
+---------+----------+-----------+-------------+------+----------+-------------+---------------+------------+------------+--------+-------+--------+-------+
|Order_ID |Order_Date|Customer_ID|Customer_Name|Region|State     |City         |Category       |Sub_Category|Product_Name|Quantity|Sales  |Discount|Profit |
+---------+----------+-----------+-------------+------+----------+-------------+---------------+------------+------------+--------+-------+--------+-------+
|ORD000001|2023-12-30|CUST8597   |Customer 58  |South |Illinois  |Los Angeles  |Office Supplies|Copiers     |Wood Table  |18      |834.15 |0.15    |42.58  |
|ORD000002|2023-10-21|CUST8456   |Customer 236 |East  |California|San Francisco|Technology     |Phones      |Smartphone  |14      |1845.72|0.1     |18.88  |
|ORD000003|2023-03-31|CUST9415   |Customer 12  |West  |Georgia   |Miami        |Technology     |Phones      |Printer     |15      |2465.02|0.1     |-39.79 |
|ORD000004|2023-08-04|CUST1326   |Custo

## 14. Wide vs Narrow Transformations

Narrow transformations do not move data between partitions. Wide transformations do and may cause shuffle.

Example narrow: `select()`, `filter()`.
Example wide: `groupBy()`, `join()`, `distinct()`.

Shuffle diagram:
```
Partition 1 ----
Partition 2 ----> Shuffle ----> Partition 3
Partition 4 ----
```

## 15. Predicate Pushdown

Predicate pushdown helps Spark read only relevant rows from Parquet files. This improves performance.

In [103]:
# Read and filter data - Spark uses predicate pushdown for performance
predicate_df = spark.read.csv(csv_file, header=True, inferSchema=True)
predicate_df = predicate_df.select('Region', 'Sales', 'Profit')
predicate_df = predicate_df.filter(predicate_df['Sales'] > 2000)

print("Rows with Sales > 2000:")
predicate_df.show(5, truncate=False)
print('Total rows after filter:', predicate_df.count())

Rows with Sales > 2000:
+------+-------+-------+
|Region|Sales  |Profit |
+------+-------+-------+
|West  |2465.02|-39.79 |
|West  |3902.09|805.28 |
|East  |2980.62|-101.89|
|North |2328.49|-49.41 |
|North |4250.64|-355.47|
+------+-------+-------+
only showing top 5 rows

Total rows after filter: 6144


## 16. CSV vs Parquet Comparison

| Aspect | CSV | Parquet |
|---|---|---|
| Storage | Text row-based | Columnar binary |
| Compression | No built-in compression | Built-in compression |
| Performance | Slower for analytics | Faster for analytics |
| Schema | No schema stored | Schema is stored |
| Read Speed | Slower | Faster |
| Write Speed | Simple | Good for analytics |
| Predicate Pushdown | Limited | Strong |
| Best Use Case | Simple files | Data lake and analytics |


## 17. Build Complete Pipeline

Read CSV, clean data, filter, compute metrics, and save results.

In [104]:
# Step 1: Read CSV file
pipeline_df = spark.read.csv(csv_file, header=True, inferSchema=True)
print("Step 1: Read CSV file")
print(f"Total rows: {pipeline_df.count()}")

# Step 2: Clean data - remove NULL values
clean_df = pipeline_df.dropna()
print(f"\nStep 2: After removing NULL values: {clean_df.count()} rows")

# Step 3: Add Net_Sales column
clean_df = clean_df.withColumn('Net_Sales', clean_df['Sales'] * (1 - clean_df['Discount']))

# Step 4: Filter - keep only Sales > 1000
filtered_df = clean_df.filter(clean_df['Sales'] > 1000)
print(f"Step 3: After filtering (Sales > 1000): {filtered_df.count()} rows")

# Step 5: Group by Region and calculate totals
from pyspark.sql.functions import sum as spark_sum, avg as spark_avg

result_df = filtered_df.groupBy('Region').agg(
    spark_sum('Sales').alias('Total_Sales'),
    spark_avg('Sales').alias('Average_Sales'),
    spark_sum('Profit').alias('Total_Profit')
)

print("\nStep 4: Results by Region:")
result_df.show(10, truncate=False)

Step 1: Read CSV file
Total rows: 10000

Step 2: After removing NULL values: 10000 rows
Step 3: After filtering (Sales > 1000): 8092 rows

Step 4: Results by Region:
+------+-----------------+------------------+------------------+
|Region|Total_Sales      |Average_Sales     |Total_Profit      |
+------+-----------------+------------------+------------------+
|West  |6191207.510000003|3008.3612779397486|277982.3599999994 |
|North |6282707.84       |3033.6590246257847|296948.22000000015|
|East  |5892451.419999997|2974.4833013629463|287560.22000000003|
|South |5999833.690000001|3027.161296670031 |281903.16999999963|
+------+-----------------+------------------+------------------+



## 18. Save Output

The final aggregated results are saved in both CSV and Parquet.

## 19. Performance Best Practices

- Avoid `collect()` on large datasets.
- Use `show()` for small samples.
- Filter early.
- Select only needed columns.
- Use Parquet for analytics.
- Avoid unnecessary shuffles.
- Use built-in Spark functions.

## 20. Execution Results

Show the result schema and top rows.

In [105]:
# Display the final result
print("Pipeline Result")
result_df.printSchema()
result_df.show(10, truncate=False)


Pipeline Result
root
 |-- Region: string (nullable = true)
 |-- Total_Sales: double (nullable = true)
 |-- Average_Sales: double (nullable = true)
 |-- Total_Profit: double (nullable = true)

+------+-----------------+------------------+------------------+
|Region|Total_Sales      |Average_Sales     |Total_Profit      |
+------+-----------------+------------------+------------------+
|West  |6191207.510000003|3008.3612779397486|277982.3599999994 |
|North |6282707.84       |3033.6590246257847|296948.22000000015|
|East  |5892451.419999997|2974.4833013629463|287560.22000000003|
|South |5999833.690000001|3027.161296670031 |281903.16999999963|
+------+-----------------+------------------+------------------+



## 21. Insights

1. Spark uses lazy evaluation so it runs work only when needed.
2. The DAG optimizer helps Spark combine transformations.
3. Parquet is faster and smaller than CSV for analytics.
4. Filtering early reduces the data processed.
5. Predicate pushdown reads less data from Parquet.
6. Wide transformations can cause shuffle.
7. Manual schema helps avoid type issues.
8. Avoid `collect()` for large data.
9. Built-in functions are faster than Python UDFs.
10. Columnar storage is good for analytics.